# 第5章 值函数估计

## 目录1. [环境搭建](#1-环境搭建)2. [蒙特卡洛方法](#2-蒙特卡洛方法)3. [时序差分学习](#3-时序差分学习)4. [TD(λ)算法](#4-TDλ算法)5. [编程题](#5-编程题)

---

## 1. 环境搭建

In [ ]:
!pip install numpy matplotlib gym pygame torch -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import random

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("环境搭建完成！")

---

## 2. 蒙特卡洛方法

### 2.1 蒙特卡洛方法原理

蒙特卡洛（Monte Carlo, MC）方法是一类基于**经验平均**的无模型强化学习方法。

核心思想：通过对智能体与环境交互产生的多条轨迹进行采样，利用实际获得的回报的平均值来近似状态价值函数。

**首次访问MC**：仅统计第一次访问状态s时的回报
**每次访问MC**：统计所有访问状态s时的回报

In [ ]:
class MultiArmedBanditEnv:
    """多臂老虎机环境"""
    def __init__(self, n_arms=10):
        self.n_arms = n_arms
        self.probs = np.random.rand(n_arms)
    
    def reset(self):
        return 0
    
    def step(self, action):
        reward = 1 if np.random.random() < self.probs[action] else 0
        return 0, reward, True

class RandomWalkEnv:
    """随机游走环境"""
    def __init__(self, n_states=7):
        self.n_states = n_states
        self.start = n_states // 2
        self.current = self.start
        self.goal_states = [0, n_states - 1]
    
    def reset(self):
        self.current = self.start
        return self.current
    
    def step(self, action):
        if action == 0:  # 向左
            self.current = max(0, self.current - 1)
        else:  # 向右
            self.current = min(self.n_states - 1, self.current + 1)
        
        done = self.current in self.goal_states
        reward = 1 if self.current == self.n_states - 1 else 0
        return self.current, reward, done

# 测试环境
print("测试随机游走环境...")
env = RandomWalkEnv(n_states=7)
state = env.reset()
print(f"初始状态: {state}")
for _ in range(10):
    action = random.randint(0, 1)
    next_state, reward, done = env.step(action)
    print(f"  动作{action} -> 状态{next_state}, 奖励{reward}, 完成{done}")
    if done: break

### 2.2 首次访问蒙特卡洛

In [ ]:
class FirstVisitMC:
    """首次访问蒙特卡洛评估"""
    def __init__(self, n_states, gamma=0.9):
        self.n_states = n_states
        self.gamma = gamma
        self.returns_sum = np.zeros(n_states)
        self.returns_count = np.zeros(n_states)
        self.values = np.zeros(n_states)
    
    def update(self, states, rewards):
        """更新价值函数"""
        G = 0
        visited = set()
        for t in range(len(states) - 1, -1, -1):
            state = states[t]
            G = self.gamma * G + rewards[t]
            if state not in visited:
                visited.add(state)
                self.returns_sum[state] += G
                self.returns_count[state] += 1
                self.values[state] = self.returns_sum[state] / self.returns_count[state]
    
    def get_values(self):
        """获取价值函数"""
        return self.values.copy()

# 测试首次访问MC
print("测试首次访问MC...")
mc = FirstVisitMC(n_states=7)

def run_episode(env, policy):
    states = [env.reset()]
    rewards = []
    done = False
    while not done:
        action = policy(states[-1])
        next_state, reward, done = env.step(action)
        states.append(next_state)
        rewards.append(reward)
    return states[:-1], rewards

random_policy = lambda s: random.randint(0, 1)

for ep in range(100):
    states, rewards = run_episode(RandomWalkEnv(7), random_policy)
    mc.update(states, rewards)

print(f"估计价值: {mc.get_values()}")

### 2.3 增量式蒙特卡洛更新

In [ ]:
class IncrementalMC:
    """增量式蒙特卡洛评估"""
    def __init__(self, n_states, gamma=0.9):
        self.n_states = n_states
        self.gamma = gamma
        self.values = np.zeros(n_states)
        self.counts = np.zeros(n_states)
    
    def update(self, states, rewards):
        """增量式更新"""
        G = 0
        visited = set()
        for t in range(len(states) - 1, -1, -1):
            state = states[t]
            G = self.gamma * G + rewards[t]
            if state not in visited:
                visited.add(state)
                self.counts[state] += 1
                alpha = 1.0 / self.counts[state]
                self.values[state] += alpha * (G - self.values[state])
    
    def get_values(self):
        return self.values.copy()

# 对比两种MC方法
def compare_mc_methods(n_episodes=500):
    """对比MC方法"""
    env = RandomWalkEnv(7)
    
    mc1 = FirstVisitMC(7, gamma=0.9)
    mc2 = IncrementalMC(7, gamma=0.9)
    
    errors1 = []
    errors2 = []
    # 真实价值（近似）
    true_values = np.linspace(0.5, 1.0, 7)  # 近似真实值
    
    for ep in range(n_episodes):
        states, rewards = run_episode(env, random_policy)
        mc1.update(states, rewards)
        mc2.update(states, rewards)
        
        if ep % 50 == 0:
            err1 = np.mean(np.abs(mc1.get_values() - true_values))
            err2 = np.mean(np.abs(mc2.get_values() - true_values))
            errors1.append(err1)
            errors2.append(err2)
    
    plt.figure(figsize=(10, 5))
    plt.plot(errors1, label='批量式MC', linewidth=2)
    plt.plot(errors2, label='增量式MC', linewidth=2)
    plt.xlabel('回合数(×50)', fontsize=12)
    plt.ylabel('价值估计误差', fontsize=12)
    plt.title('MC方法收敛对比', fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('mc_comparison.png', dpi=150)
    plt.show()

compare_mc_methods()

---

## 3. 时序差分学习

### 3.1 TD(0)算法

时序差分（TD）学习是蒙特卡洛与动态规划的结合，通过自举法利用后续状态的价值估计来更新当前状态的价值。

TD(0)更新公式：
$$V(S_t) \leftarrow V(S_t) + \alpha \left( R_{t+1} + \gamma V(S_{t+1}) - V(S_t) \right)$$

其中：
- $R_{t+1} + \gamma V(S_{t+1})$ 是TD目标
- $\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$ 是TD误差

In [ ]:
class TD0:
    """TD(0)算法"""
    def __init__(self, n_states, alpha=0.1, gamma=0.9):
        self.n_states = n_states
        self.alpha = alpha
        self.gamma = gamma
        self.values = np.zeros(n_states)
    
    def update(self, state, reward, next_state, done):
        """TD(0)更新"""
        if done:
            target = reward
        else:
            target = reward + self.gamma * self.values[next_state]
        td_error = target - self.values[state]
        self.values[state] += self.alpha * td_error
    
    def get_values(self):
        return self.values.copy()

# 测试TD(0)
print("测试TD(0)...")
td = TD0(n_states=7, alpha=0.1, gamma=0.9)
env = RandomWalkEnv(7)

def td_episode(env, values, epsilon=0.1):
    """TD(0)回合"""
    state = env.reset()
    done = False
    while not done:
        if random.random() < epsilon:
            action = random.randint(0, 1)
        else:
            action = 1 if values[state + 1] > values[state - 1] else 0 if state > 0 else 1
        next_state, reward, done = env.step(action)
        td.update(state, reward, next_state, done)
        state = next_state
        if done: break

for _ in range(500):
    td_episode(env, td.values)

print(f"TD(0)估计价值: {td.get_values()}")

### 3.2 MC与TD对比

In [ ]:
def compare_mc_td(n_episodes=500):
    """对比MC与TD"""
    mc = IncrementalMC(7, gamma=0.9)
    td = TD0(7, alpha=0.1, gamma=0.9)
    
    env = RandomWalkEnv(7)
    true_values = np.linspace(0.5, 1.0, 7)
    
    mc_errors = []
    td_errors = []
    
    for ep in range(n_episodes):
        states, rewards = run_episode(env, random_policy)
        mc.update(states, rewards)
        td_episode(env, td.values)
        
        if ep % 50 == 0:
            mc_errors.append(np.mean(np.abs(mc.get_values() - true_values)))
            td_errors.append(np.mean(np.abs(td.get_values() - true_values)))
    
    plt.figure(figsize=(10, 5))
    plt.plot(mc_errors, label='蒙特卡洛', linewidth=2)
    plt.plot(td_errors, label='TD(0)', linewidth=2)
    plt.xlabel('回合数(×50)', fontsize=12)
    plt.ylabel('价值估计误差', fontsize=12)
    plt.title('MC vs TD(0) 收敛对比', fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('mc_vs_td.png', dpi=150)
    plt.show()

compare_mc_td()

---

## 4. TD(λ)算法

### 4.1 TD(λ)原理

**TD(λ)**通过资格迹机制融合多步信息，在偏差与方差之间取得平衡。

n步回报：
$$G_t^{(n)} = R_{t+1} + \gamma R_{t+2} + \dots + \gamma^{n-1} R_{t+n} + \gamma^n V(S_{t+n})$$

λ-回报：
$$G_t^{\lambda} = (1-\lambda) \sum_{n=1}^{\infty} \lambda^{n-1} G_t^{(n)}$$

λ=0 时等同于TD(0)，λ=1 时等同于MC

In [ ]:
class TDLambda:
    """TD(λ)算法"""
    def __init__(self, n_states, alpha=0.1, gamma=0.9, lamda=0.9):
        self.n_states = n_states
        self.alpha = alpha
        self.gamma = gamma
        self.lamda = lamda
        self.values = np.zeros(n_states)
        self.eligibility = np.zeros(n_states)
    
    def reset(self):
        """回合开始重置资格迹"""
        self.eligibility = np.zeros(self.n_states)
    
    def update(self, state, reward, next_state, done):
        """TD(λ)更新"""
        if done:
            target = reward
        else:
            target = reward + self.gamma * self.values[next_state]
        
        td_error = target - self.values[state]
        self.eligibility[state] += 1
        self.values += self.alpha * td_error * self.eligibility
        self.eligibility *= self.gamma * self.lamda
    
    def get_values(self):
        return self.values.copy()

# 测试TD(λ)
print("测试TD(λ)...")
for lamda in [0.0, 0.3, 0.6, 0.9]:
    td = TDLambda(7, alpha=0.1, gamma=0.9, lamda=lamda)
    env = RandomWalkEnv(7)
    
    for _ in range(500):
        td.reset()
        state = env.reset()
        done = False
        while not done:
            action = random.randint(0, 1)
            next_state, reward, done = env.step(action)
            td.update(state, reward, next_state, done)
            state = next_state
            if done: break
    
    print(f"  λ={lamda}: {td.get_values()}")

### 4.2 不同λ值对比

In [ ]:
def compare_lambda(n_episodes=500):
    """对比不同λ值"""
    lambdas = [0.0, 0.3, 0.6, 0.9, 1.0]
    colors = ['blue', 'orange', 'green', 'red', 'purple']
    
    env = RandomWalkEnv(7)
    true_values = np.linspace(0.5, 1.0, 7)
    
    plt.figure(figsize=(12, 5))
    
    for lamda, color in zip(lambdas, colors):
        errors = []
        td = TDLambda(7, alpha=0.1, gamma=0.9, lamda=lamda)
        
        for ep in range(n_episodes):
            td.reset()
            state = env.reset()
            done = False
            while not done:
                action = random.randint(0, 1)
                next_state, reward, done = env.step(action)
                td.update(state, reward, next_state, done)
                state = next_state
                if done: break
            
            if ep % 50 == 0:
                errors.append(np.mean(np.abs(td.get_values() - true_values)))
        
        plt.plot(errors, label=f'λ={lamda}', color=color, linewidth=2)
    
    plt.xlabel('回合数(×50)', fontsize=12)
    plt.ylabel('价值估计误差', fontsize=12)
    plt.title('不同λ值对比', fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('lambda_comparison.png', dpi=150)
    plt.show()

compare_lambda()

---

## 5. 编程题

### 编程题1

**实现首次访问型蒙特卡洛评估算法，在随机游走环境上测试算法收敛性，观察不同回合数对价值估计精度的影响。**

In [ ]:
"""
编程题1：首次访问MC评估
"""
def program_exercise_1():
    """编程题1：MC收敛性"""
    n_episodes_list = [100, 500, 1000, 5000]
    env = RandomWalkEnv(7)
    
    plt.figure(figsize=(12, 5))
    
    for n_episodes in n_episodes_list:
        mc = IncrementalMC(7, gamma=0.9)
        
        for _ in range(n_episodes):
            states, rewards = run_episode(env, random_policy)
            mc.update(states, rewards)
        
        plt.subplot(1, 2, 1)
        plt.plot(mc.get_values(), label=f'{n_episodes}回合', linewidth=2)
    
    plt.xlabel('状态', fontsize=12)
    plt.ylabel('价值估计', fontsize=12)
    plt.title('不同回合数的价值估计', fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # 收敛曲线
    errors = []
    mc = IncrementalMC(7, gamma=0.9)
    true_values = np.linspace(0.5, 1.0, 7)
    
    for ep in range(5000):
        states, rewards = run_episode(env, random_policy)
        mc.update(states, rewards)
        if ep % 100 == 0:
            errors.append(np.mean(np.abs(mc.get_values() - true_values)))
    
    plt.subplot(1, 2, 2)
    plt.plot(errors, linewidth=2)
    plt.xlabel('回合数(×100)', fontsize=12)
    plt.ylabel('误差', fontsize=12)
    plt.title('MC收敛曲线', fontsize=14)
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('pe1.png', dpi=150)
    plt.show()

program_exercise_1()

### 编程题2

**实现TD(0)算法，与蒙特卡洛方法在相同环境下进行对比实验，分析两种方法的收敛速度和估计精度的差异。**

In [ ]:
"""
编程题2：TD(0) vs MC
"""
def program_exercise_2():
    """编程题2：TD(0) vs MC"""
    n_episodes = 1000
    env = RandomWalkEnv(7)
    true_values = np.linspace(0.5, 1.0, 7)
    
    mc = IncrementalMC(7, gamma=0.9)
    td = TD0(7, alpha=0.1, gamma=0.9)
    
    mc_errors = []
    td_errors = []
    
    for ep in range(n_episodes):
        states, rewards = run_episode(env, random_policy)
        mc.update(states, rewards)
        td_episode(env, td.values)
        
        if ep % 20 == 0:
            mc_errors.append(np.mean(np.abs(mc.get_values() - true_values)))
            td_errors.append(np.mean(np.abs(td.get_values() - true_values)))
    
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(mc.get_values(), label='MC', linewidth=2)
    plt.plot(td.get_values(), label='TD(0)', linewidth=2)
    plt.plot(true_values, 'k--', label='真实值', linewidth=2)
    plt.xlabel('状态', fontsize=12)
    plt.ylabel('价值', fontsize=12)
    plt.title('价值函数估计对比', fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    plt.plot(mc_errors, label='MC', linewidth=2)
    plt.plot(td_errors, label='TD(0)', linewidth=2)
    plt.xlabel('回合数(×20)', fontsize=12)
    plt.ylabel('误差', fontsize=12)
    plt.title('收敛速度对比', fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('pe2.png', dpi=150)
    plt.show()

program_exercise_2()

### 编程题3

**实现TD(λ)算法，通过参数λ控制多步回报的权重，观察不同λ值对算法性能的影响。**

In [ ]:
"""
编程题3：TD(λ)参数分析
"""
def program_exercise_3():
    """编程题3：TD(λ)分析"""
    lambdas = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    n_episodes = 1000
    env = RandomWalkEnv(7)
    true_values = np.linspace(0.5, 1.0, 7)
    
    results = {}
    
    plt.figure(figsize=(14, 5))
    
    for lamda in lambdas:
        td = TDLambda(7, alpha=0.1, gamma=0.9, lamda=lamda)
        errors = []
        
        for ep in range(n_episodes):
            td.reset()
            state = env.reset()
            done = False
            while not done:
                action = random.randint(0, 1)
                next_state, reward, done = env.step(action)
                td.update(state, reward, next_state, done)
                state = next_state
                if done: break
            
            if ep % 20 == 0:
                errors.append(np.mean(np.abs(td.get_values() - true_values)))
        
        results[lamda] = errors[-1]
        plt.plot(errors, label=f'λ={lamda}', linewidth=2)
    
    plt.xlabel('回合数(×20)', fontsize=12)
    plt.ylabel('价值估计误差', fontsize=12)
    plt.title('TD(λ)不同λ值性能对比', fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('pe3.png', dpi=150)
    plt.show()
    
    print("\n最终误差汇总：")
    for lamda, error in results.items():
        print(f"  λ={lamda}: {error:.4f}")

program_exercise_3()

### 编程题4

**修改学习率α，观察收敛速度的变化；修改折扣因子γ，比较不同γ设置下的价值估计差异。**

In [ ]:
"""
编程题4：超参数分析
"""
def program_exercise_4():
    """编程题4：超参数分析"""
    n_episodes = 1000
    env = RandomWalkEnv(7)
    true_values = np.linspace(0.5, 1.0, 7)
    
    print("="*50)
print("1. 学习率α分析")
print("="*50)
    
    alphas = [0.01, 0.05, 0.1, 0.2, 0.5]
    alpha_results = {}
    
    for alpha in alphas:
        td = TD0(7, alpha=alpha, gamma=0.9)
        errors = []
        
        for ep in range(n_episodes):
            td_episode(env, td.values)
            if ep % 50 == 0:
                errors.append(np.mean(np.abs(td.get_values() - true_values)))
        
        alpha_results[alpha] = errors[-1]
        print(f"  α={alpha}: 最终误差={errors[-1]:.4f}")
    
    print("\n" + "="*50)
    print("2. 折扣因子γ分析")
    print("="*50)
    
    gammas = [0.5, 0.7, 0.9, 0.95, 0.99]
    gamma_results = {}
    
    for gamma in gammas:
        td = TD0(7, alpha=0.1, gamma=gamma)
        values = []
        
        for ep in range(n_episodes):
            td_episode(env, td.values)
            if ep % 50 == 0:
                values.append(td.get_values()[3])
        
        gamma_results[gamma] = values[-1]
        print(f"  γ={gamma}: 中心状态价值={values[-1]:.4f}")
    
    # 可视化
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    axes[0].bar(range(len(alphas)), list(alpha_results.values()))
    axes[0].set_xticks(range(len(alphas)))
    axes[0].set_xticklabels([str(a) for a in alphas])
    axes[0].set_xlabel('学习率α', fontsize=12)
    axes[0].set_ylabel('最终误差', fontsize=12)
    axes[0].set_title('学习率影响', fontsize=14)
    axes[0].grid(True, alpha=0.3)
    
    axes[1].bar(range(len(gammas)), list(gamma_results.values()))
    axes[1].set_xticks(range(len(gammas)))
    axes[1].set_xticklabels([str(g) for g in gammas])
    axes[1].set_xlabel('折扣因子γ', fontsize=12)
    axes[1].set_ylabel('中心状态价值', fontsize=12)
    axes[1].set_title('折扣因子影响', fontsize=14)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('pe4.png', dpi=150)
    plt.show()

program_exercise_4()

In [ ]:
print("\n" + "="*50)
print("第5章 值函数估计 - 内容完成！")
print("="*50)